AMUSE tutorial on particle sets
====================


This tutorial introduces you the the most used data store in AMUSE, a *particle set*. AMUSE particle sets are a handy tool for storing data of stellar or planetary systems.

In [24]:
from amuse.units import units, constants
from amuse.lab import Particles, Particle

A particle set store the properties of a system under study, for example:

In [25]:
sun_and_earth = Particles(2)

of course at this point we have not specified anything about the system, lets change that:

In [26]:
# declare the sun
sun = sun_and_earth[0]
sun.mass = 1 | units.MSun
sun.position = (0, 0, 0) | units.au
sun.velocity = (0, 0, 0) | units.kms
print(f"Sun={sun}")

Sun=Particle(16839529552159934374, set=<4698226768>
    , mass=1.0 MSun
    , vx=0.0 kms
    , vy=0.0 kms
    , vz=0.0 kms
    , x=0.0 au
    , y=0.0 au
    , z=0.0 au)


In [27]:
# now declare the Earth
def relative_orbital_velocity(mass, distance):
    return (constants.G * mass / distance).sqrt()

earth = sun_and_earth[1]
earth.mass = 1 | units.MEarth
earth.position = (1, 0, 0) | units.au

vorb = relative_orbital_velocity(
    sun_and_earth.total_mass(), 
    earth.position.length()
)
earth.velocity = (0, 1, 0) * vorb
print(f"Earth={earth}")

Earth=Particle(14722465344295373437, set=<4698226768>
    , mass=3.00273515275e-06 MSun
    , vx=0.0 kms
    , vy=29.7885123292 kms
    , vz=0.0 kms
    , x=1.0 au
    , y=0.0 au
    , z=0.0 au)


In [28]:
# normally it is a good idea to study a system in its center of mass:
sun_and_earth.move_to_center()
print(sun_and_earth)

                 key         mass           vx           vy           vz            x            y            z
                   -         MSun          kms          kms          kms           au           au           au
====================  ===========  ===========  ===========  ===========  ===========  ===========  ===========
16839529552159934374    1.000e+00    0.000e+00   -8.945e-05    0.000e+00   -3.003e-06    0.000e+00    0.000e+00
14722465344295373437    3.003e-06    0.000e+00    2.979e+01    0.000e+00    1.000e+00    0.000e+00    0.000e+00
====================  ===========  ===========  ===========  ===========  ===========  ===========  ===========


Now, let's give the particles a specific name (or other attribute)

In [29]:
sun_and_earth.name = ["sun", "earth"]

How we have declared the particles and moved them to the center of mass.
We can also search for a specific particle.
For example, the one with the "earth" in the attribute "name".

In [30]:
earth = sun_and_earth[sun_and_earth.name=="earth"][0] # note the index! do you understand the difference when removing it?
print(f"Earth={earth}")

Earth=Particle(14722465344295373437, set=<4698226768>
    , mass=3.00273515275e-06 MSun
    , name=earth
    , vx=0.0 kms
    , vy=29.7884228825 kms
    , vz=0.0 kms
    , x=0.999996997274 au
    , y=0.0 au
    , z=0.0 au)


We can add a moon in orbit around the earth

In [31]:
moon = Particle()
moon.name = "moon"
moon.mass = 7.34767309e+22 | units.kg
moon.position = (384400, 0, 0) | units.km
vorb = relative_orbital_velocity(
    earth.mass + moon.mass, 
    moon.position.length()
)
moon.velocity = (0, 1, 0) * vorb
print(f"moon={moon}")

moon=Particle(2620547091991495362, set=<5135786832>
    , mass=7.34767309e+22 kg
    , name=moon
    , vx=0.0 44597309335878.1 * m * s**-1
    , vy=2.2973349771e-11 44597309335878.1 * m * s**-1
    , vz=0.0 44597309335878.1 * m * s**-1
    , x=384400.0 km
    , y=0.0 km
    , z=0.0 km)


The Moon, however, is now somewhere inside the Sun with zero velocity - which is not good.
We will have to relocate the Moon to make it orbit around the Earth.
We do that by simply adding the positions and velocity of Earth to the Moon's.

In [32]:
moon.position += earth.position
moon.velocity += earth.velocity

And we can add the moon to the Sun and Earth system

In [33]:
sun_and_earth.add_particle(moon)

Note the use of the singular here, because we only add a single particle to the particle set sun_and_earth.
It is probably better to rename the sun_and_earth now.

In [34]:
solar_system = sun_and_earth

It is important now to recenter the entire system, because by adding the moon we shifted the center of mass.

In [35]:
solar_system.move_to_center()
print(f"Solar system:\n{solar_system}")

Solar system:
                 key         mass         name           vx           vy           vz            x            y            z
                   -         MSun         none          kms          kms          kms           au           au           au
====================  ===========  ===========  ===========  ===========  ===========  ===========  ===========  ===========
16839529552159934374    1.000e+00          sun    0.000e+00   -9.059e-05    0.000e+00   -3.040e-06    0.000e+00    0.000e+00
14722465344295373437    3.003e-06        earth    0.000e+00    2.979e+01    0.000e+00    1.000e+00    0.000e+00    0.000e+00
 2620547091991495362    3.694e-08         moon    0.000e+00    3.081e+01    0.000e+00    1.003e+00    0.000e+00    0.000e+00
====================  ===========  ===========  ===========  ===========  ===========  ===========  ===========  ===========


The key uniquely identifies a particle, but is otherwise never explicitly used:

In [36]:
print(moon in solar_system)

True


We can now manipulate the planetary system, or query it.
for example by querying the masses.

In [37]:
print("mass=", solar_system.mass.in_(units.MEarth))

mass= [333029.704297, 1.0, 0.0123031263019] MEarth


This gives us a list of the masses of all objects, in units the the earth's mass. In fact, each of the particle's attributes is a simple numpy array: it can be assigned and manipulated as such.

Another way to acquire the same information could be done as follows:

In [38]:
print(f"mass={solar_system.mass/solar_system[1].mass}")

mass=[  3.33029704e+05   1.00000000e+00   1.23031263e-02]


Note that we get a reference to the attributes..lets make a copy of the solar system and play around with it:

In [39]:
solar_system2=solar_system.copy()
x=solar_system2.x
print(x is solar_system2.x)
print(solar_system2.center_of_mass())
x[0]+=0.1| units.AU
print(solar_system2.center_of_mass())

True
[-5.95568230702e-23, 0.0, 0.0] 149597870691.0 * m
[0.0999996960331, 0.0, 0.0] 149597870691.0 * m


But this is not necessarily true if the particle set is stored in a code, and can not be relied upon!

In some (hopefully rare) cases you may want to use the particle set or its attributes as simple numpy arrays, without the units.
This is easily achieved by stripping the unit from the array. This can be realied by explicitely querying the selected parameter with that specific unit.

In [40]:
solar_system.position.value_in(units.parsec)

array([[ -1.47371911e-11,   0.00000000e+00,   0.00000000e+00],
       [  4.84812207e-06,   0.00000000e+00,   0.00000000e+00],
       [  4.86057963e-06,   0.00000000e+00,   0.00000000e+00]])

which, in this case, gives you a 2-dimensional *numpy.array* (a copy!) of the positions of star, planet and moon in units of a parsec.

Now, you may want to query the particle set solar_system.
For example by asking what attributes it has.
This can be done as follows:

In [41]:
dir(solar_system)

['GLOBAL_DERIVED_ATTRIBUTES',
 'LagrangianRadii',
 'Qparameter',
 '__add__',
 '__annotate_func__',
 '__annotations_cache__',
 '__array__',
 '__array_interface__',
 '__array_struct__',
 '__class__',
 '__contains__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__or__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__sub__',
 '__subclasshook__',
 '__weakref__',
 '_add_indices_in_attribute_storage',
 '_as_masked_subset_in',
 '_attributes_for_dir',
 '_convert_from_entities_or_quantities',
 '_convert_to_entities_or_quantities',
 '_convert_to_entities_or_quantities_async',
 '_convert_to_entity_or_quantity',
 '_factory_for_new_collection',
 '_get_

Or get some general help on the underlying particle class

In [42]:
help(solar_system)

Help on Particles in module amuse.datamodel.particles object:

class Particles(AbstractParticleSet)
 |  Particles(
 |      size=0,
 |      storage=None,
 |      keys=None,
 |      keys_generator=None,
 |      particles=None,
 |      is_working_copy=True,
 |      **attributes
 |  )
 |
 |  A set of particles. Attributes and values are stored in
 |  a private storage model. This storage model can store
 |  the values in the python memory space, in the memory space
 |  of the code or in a HDF5 file. By default the storage
 |  model is in memory.
 |
 |  Method resolution order:
 |      Particles
 |      AbstractParticleSet
 |      amuse.datamodel.base.AbstractSet
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __getitem__(self, index)
 |
 |  __init__(
 |      self,
 |      size=0,
 |      storage=None,
 |      keys=None,
 |      keys_generator=None,
 |      particles=None,
 |      is_working_copy=True,
 |      **attributes
 |  )
 |      Initialize self.  See help(type(self)) for

You have performed some rudimentary operations on a particle set.
It is now time to experiment a little for yourself.


Assignments and questions:
---------------

### Assignment 1:
Add the planet Jupiter (see [Wikipedia](https://en.wikipedia.org/wiki/Jupiter)) to your small planetary system.

In [43]:
jupiter = Particle()
jupiter.name = "jupiter"
jupiter.mass = 1.898e+27 | units.kg
jupiter.position = (384400, 0, 0) | units.km
vorb = relative_orbital_velocity(
    earth.mass + jupiter.mass, 
    jupiter.position.length()
)
jupiter.velocity = (0, 1, 0) * vorb
print(f"jupiter={jupiter}")

jupiter.position += earth.position
jupiter.velocity += earth.velocity

sun_and_earth.add_particle(jupiter)

jupiter=Particle(14556322841463877777, set=<5135724816>
    , mass=1.898e+27 kg
    , name=jupiter
    , vx=0.0 44597309335878.1 * m * s**-1
    , vy=4.0769197758e-10 44597309335878.1 * m * s**-1
    , vz=0.0 44597309335878.1 * m * s**-1
    , x=384400.0 km
    , y=0.0 km
    , z=0.0 km)


### Assignment 2:
Your planetary system is notoriously planar, and initialy the earth and moon are positioned along the Cartesian x-axis with the velocity vector in the Cartesian y-direction. 

Make the Sun-Earth-Moon system more realistic by introducing a small inclination to the Earth's and Moon's orbits and by giving them a random mean anomaly.

The [Orbital element module](https://github.com/amusecode/amuse/blob/main/src/amuse/ext/orbital_elements.py) of AMUSE could come in handy. 

In [61]:
from amuse.ext.orbital_elements import new_binary_from_orbital_elements
binary=new_binary_from_orbital_elements(1| units.MSun, 1| units.MEarth, semimajor_axis =1| units.AU, eccentricity=0.)

In [62]:
import numpy as np
rng = np.random.default_rng()

sun_earth = new_binary_from_orbital_elements(1 | units.MSun, 1 | units.MEarth, semimajor_axis =1 | units.AU, eccentricity=0.0167, 
    inclination=(5.0 * np.pi / 180.0), longitude_of_the_ascending_node=0.0, argument_of_periapsis=0.0,
    true_anomaly=rng.uniform(0.0, 2.0 * np.pi)
)

sun = sun_earth[0]
earth = sun_earth[1]

In [63]:
earth_moon = new_binary_from_orbital_elements(1 | units.MEarth, 1 | units.MEarth, semimajor_axis =1 | units.AU, eccentricity=0.0167, 
    inclination=(5.0 * np.pi / 180.0), longitude_of_the_ascending_node=0.0, argument_of_periapsis=0.0,
    true_anomaly=rng.uniform(0.0, 2.0 * np.pi)
)

moon = earth_moon[1]

In [64]:
moon.position += earth.position
moon.velocity += earth.velocity

solar_system = Particles()
solar_system.add_particle(sun)
solar_system.add_particle(earth)
solar_system.add_particle(moon)

solar_system.move_to_center()

print(solar_system)

                 key         mass           vx           vy           vz            x            y            z
                   -  1.98892e+30 * kg  3646245880.3572216 * m * s**-1  3646245880.3572216 * m * s**-1  3646245880.3572216 * m * s**-1  149597870691.0 * m  149597870691.0 * m  149597870691.0 * m
====================  ===========  ===========  ===========  ===========  ===========  ===========  ===========
14620609968137726657    1.000e+00    2.182e-12   -4.987e-11   -7.596e-14   -7.369e-06   -4.088e-07   -6.226e-10
13271098965515602286    3.003e-06   -3.629e-07    8.299e-06    1.264e-08    9.823e-01    4.367e-02    6.652e-05
 9133813434965179516    3.003e-06   -3.639e-07    8.309e-06    1.266e-08    1.472e+00    9.246e-02    1.408e-04
====================  ===========  ===========  ===========  ===========  ===========  ===========  ===========


### Assignment 3:
Calculate the total  gravitational binding energy of solarsystem.

Now displace the entire particle set by 100 parsec and give it a linear velocity of 100km/s in the z-direction.
Then calculate the binding energy of the system again.

Did the binding energy of the Solar system change by this translation?

In [65]:
E_before = solar_system.potential_energy() + solar_system.kinetic_energy()
print("E_before =", E_before)

solar_system.position += (100, 0, 0) | units.parsec

solar_system.velocity += (0, 0, 100) | units.km / units.s

E_after = solar_system.potential_energy() + solar_system.kinetic_energy()
print("E_after =", E_after)
print("Delta E =", E_after - E_before)

E_before = -1.32264686441e-16 2.644290823210217e+49 * m**2 * kg * s**-2
E_after = 3.76080275938e-10 2.644290823210217e+49 * m**2 * kg * s**-2
Delta E = 3.76080408203e-10 2.644290823210217e+49 * m**2 * kg * s**-2


### Question 1:
Particle sets have the attribute *get_binaries()*.
If you use this function to check the binaries in your system you will find that (without Jupiter) you have 3 binaries. Explain why the Sun is in a binary with the Moon. You may want to take a look at the [source code](https://github.com/amusecode/amuse/blob/main/src/amuse/datamodel/particle_attributes.py).

because its not showing graviational binaries, its just all pairs of two particles in the particle set and because the moon is in the same particle set as the sun and the earth, its in a binary with each of them and they are in a binary with each other

### Assignment 4:
Generate another particle set with a 2 solar-mass star and two planets of 10 and 100 Earth masses in circular orbits at 0.1 and 0.6 au.
Place this second planetary system at apocenter around your Solar system (true anomaly of 180 degrees) at a semimajor axis of 60 au with an eccentricity of 0.6.
Then move the entire system to the center of mass.

In [66]:
new_system = Particles(3)


star = new_system[0]
star.mass = 2 | units.MSun
star.position = (0, 0, 0) | units.au
star.velocity = (0, 0, 0) | units.kms

planet1 = new_system[1]
planet1.mass = 10 | units.MEarth
planet1.position = (0.1, 0, 0) | units.au
planet1.velocity = (0, 30, 0) | units.kms

planet2 = new_system[2]
planet2.mass = 100 | units.MEarth
planet2.position = (0.6, 0, 0) | units.au
planet2.velocity = (0, 24, 0) | units.kms

In [68]:
star_planet1 = new_binary_from_orbital_elements(2 | units.MSun, 10 | units.MEarth, semimajor_axis = 60 | units.AU, eccentricity=0.0167, 
    inclination=(5.0 * np.pi / 180.0), longitude_of_the_ascending_node=0.0, argument_of_periapsis=0.0,
    true_anomaly=180
)

star_planet2 = new_binary_from_orbital_elements(2 | units.MSun, 100 | units.MEarth, semimajor_axis = 60 | units.AU, eccentricity=0.0167, 
    inclination=(5.0 * np.pi / 180.0), longitude_of_the_ascending_node=0.0, argument_of_periapsis=0.0,
    true_anomaly=180
)

star = star_planet1[0]
planet1 = star_planet1[1]
planet2 = star_planet2[1]

new_system = Particles()
new_system.add_particle(star)
new_system.add_particle(planet1)
new_system.add_particle(planet2)
new_system.move_to_center()

### Question 2:
Which of the orbits of the binary star with planets from *Assignment 4* has the highest binding energy?

In [71]:
orbit1_binding_energy = star_planet1.potential_energy() + star_planet1.kinetic_energy()
orbit2_binding_energy = star_planet2.potential_energy() + star_planet2.kinetic_energy()

print("binding energy orbit 1 (star-planet1) =", orbit1_binding_energy)
print("binding energy orbit 2 (star-planet2) =", orbit2_binding_energy)

if orbit1_binding_energy > orbit2_binding_energy:
    print("Orbit 1 (star-planet1) has the highest binding energy.")
elif orbit2_binding_energy > orbit1_binding_energy:
    print("Orbit 2 (star-planet2) has the highest binding energy.")
else:
    print("Both orbits have the same binding energy.")

binding energy orbit 1 (star-planet1) = -3.34018252921e-17 2.644290823210217e+49 * m**2 * kg * s**-2
binding energy orbit 2 (star-planet2) = -3.34018252921e-16 2.644290823210217e+49 * m**2 * kg * s**-2
Orbit 1 (star-planet1) has the highest binding energy.


## Appendix: the Particle attributes

particle sets carry a predefined set of methods and calculated attributes - which may save you some work! Below, we listed some of the most useful, you are encouraged to experiment
to see if you can understand what they do.

In [72]:
from amuse.ic.plummer import new_plummer_model
p=new_plummer_model(100)

The following provide particle quantities (some of these are aliases):
`position`
`velocity`
`acceleration`
`potential`
`specific_kinetic_energy`
`angular_momentum`

The following provide information about the physical state of whole particle set:
`center_of_mass`
`center_of_mass_velocity`
`kinetic_energy`
`potential_energy`
`virial_radius`
`total_mass`
`total_radius`
`total_momentum`
`total_angular_momentum`
`moment_of_inertia`
`dynamical_timescale`

The following change the particle set:
`move_to_center`
`scale_to_standard`
`rotate`
`add_spin`

The following are (mainly) used for the analysis of star clusters:
`get_binaries`
`densitycentre_coreradius_coredens`
`cluster_core`
`bound_subset`
`LagrangianRadii`
`mass_segregation_Gini_coefficient`

The following provides some more miscelleaneous analysis tools:
`find_closest_particle_to`
`distances_squared`
`nearest_neighbour`
`Qparameter`
`connected_components`
`minimum_spanning_tree_length`
`mass_segregation_ratio`
`mass_segregation_ratio_from_nearest_neighbour`
`correlation_dimension`
`box_counting_dimension`


In [73]:
print(p.virial_radius())
cm=p.center_of_mass()
print(p.LagrangianRadii(cm=cm))

0.984496153014 length
(quantity<[0.136445365683, 0.140659989219, 0.179600252748, 0.284175791381, 0.377186942231, 0.753455492074, 1.4746172963, 2.12973973085, 6.47804771013] length>, [0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 0.75, 0.9, 1])
